# SAP-RPT invoice run from BTP

## Goal

Send the shared masked invoice request to your SAP AI Core deployment, then save the raw response, 128 scored predictions, and a provenance record. The notebook uses `scripts/run_rpt16_invoices.py`. A live request can incur SAP inference charges.

The runner reads a repository-root `.env` file automatically. It accepts either a direct `SAP_RPT_DEPLOYMENT_URL` and `SAP_RPT_AUTH_TOKEN`, or the SAP AI Core service-key settings `SAP_AI_CORE_API_URL`, `SAP_AI_CORE_AUTH_URL`, `SAP_AI_CORE_CLIENT_ID`, `SAP_AI_CORE_CLIENT_SECRET`, and `SAP_RPT_DEPLOYMENT_ID`. `SAP_AI_CORE_RESOURCE_GROUP` and `SAP_RPT_MODEL_NAME` are optional. The model name is a configured label, not independent proof of the deployed model version. The `.env` file is gitignored; never print its credentials.

## 1. Validate the request and shared split

The first 1,024 rows provide labeled context. Both targets in the final 128 rows must be `[PREDICT]`, so their true labels are not sent for inference.

In [1]:
import hashlib
import json
import os
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

ROOT = next(path for path in (Path.cwd(), Path.cwd().parent) if (path / "data/synthetic_invoices/payment_behavior.csv").exists())
sys.path.insert(0, str(ROOT / "scripts"))
from generate_synthetic_invoices import CONTEXT_ROWS, PREDICTION_ROWS, TARGETS
from run_rpt16_invoices import DATA, has_rpt_configuration, load_local_env, run_experiment
load_local_env()

source = pd.read_csv(DATA / "payment_behavior.csv")
request_path = DATA / "rpt16_request.json"
request = json.loads(request_path.read_text(encoding="utf-8"))
request_rows = pd.DataFrame(request["rows"])
assert len(source) == len(request_rows) == CONTEXT_ROWS + PREDICTION_ROWS
assert request["index_column"] == "invoice_id"
assert {column["name"] for column in request["prediction_config"]["target_columns"]} == set(TARGETS)
assert request_rows.invoice_id.tolist() == source.invoice_id.tolist()
feature_columns = [column for column in source.columns if column not in TARGETS]
pd.testing.assert_frame_equal(request_rows[feature_columns], source[feature_columns], check_dtype=False)
for target in TARGETS:
    assert request_rows[target].iloc[:CONTEXT_ROWS].astype(int).tolist() == source[target].iloc[:CONTEXT_ROWS].tolist()
    assert request_rows[target].iloc[CONTEXT_ROWS:].eq("[PREDICT]").all()
print(f"Validated {CONTEXT_ROWS} context and {PREDICTION_ROWS} masked test rows.")
print(f"Request SHA-256: {hashlib.sha256(request_path.read_bytes()).hexdigest()}")

Validated 1024 context and 128 masked test rows.
Request SHA-256: 822809195220d0157ee2c0976c3f4de535b10fb092a2ffa3f15bd5395bddb522


## 2. Run RPT through BTP

The `.env` file is loaded by the runner. Set `RUN_LIVE_RPT = True` only when you intend to make one live inference call. The runner obtains an OAuth token if service-key credentials are configured, retrieves the deployment URL, posts to `/predict`, and writes `rpt16_response.json`, `rpt16_results.csv`, and `rpt16_run_metadata.json`. The metadata records checksums and deployment identity without storing the client secret or token.

In [2]:
RUN_LIVE_RPT = False

if RUN_LIVE_RPT:
    if not has_rpt_configuration():
        raise RuntimeError("Configure direct RPT credentials or SAP AI Core service-key settings in .env")
    result_path = run_experiment()
    print(f"Saved {result_path.relative_to(ROOT)}")
else:
    status = "ready" if has_rpt_configuration() else "missing required settings"
    display(Markdown(f"**Live BTP call skipped.** Credential configuration: {status}. Set `RUN_LIVE_RPT` to `True` to run it."))

**Live BTP call skipped.** Credential configuration: ready. Set `RUN_LIVE_RPT` to `True` to run it.

## 3. Inspect the saved BTP result

In [3]:
results_path = DATA / "rpt16_results.csv"
metadata_path = DATA / "rpt16_run_metadata.json"
if results_path.exists() and metadata_path.exists():
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    results = pd.read_csv(results_path)
    assert metadata["source"] == "btp_deployment"
    assert metadata["request_sha256"] == hashlib.sha256(request_path.read_bytes()).hexdigest()
    assert metadata["results_sha256"] == hashlib.sha256(results_path.read_bytes()).hexdigest()
    assert results.invoice_id.tolist() == source.invoice_id.iloc[CONTEXT_ROWS:].tolist()
    for target in TARGETS:
        assert results[f"actual_{target}"].tolist() == source[target].iloc[CONTEXT_ROWS:].tolist()
    display(pd.DataFrame([{key: metadata[key] for key in ("run_at_utc", "source", "model_configured", "deployment_id", "deployment_configuration_name", "deployment_host", "resource_group", "request_sha256", "response_id", "test_rows")}]).T.rename(columns={0: "Value"}))
    display(results.head(8))
else:
    display(Markdown("**No direct BTP result is saved yet.** After a successful live run, the result and metadata will appear here."))

,Value
run_at_utc,2026-10-03T19:42:23.341603+00:00
source,btp_deployment
model_configured,sap-rpt-1.6-large
deployment_id,d8d6f47ad6f27af4
deployment_configuration_name,sap-rpt-1.6-large_autogenerated
deployment_host,api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com
resource_group,rpt-joule-orch
request_sha256,822809195220d0157ee2c0976c3f4de535b10fb092a2ff...
response_id,6d5694d3-60f4-446d-be4b-53bb0a7d45b4
test_rows,128


,invoice_id,actual_paid_late,predicted_paid_late,confidence_paid_late,actual_is_fraud,predicted_is_fraud,confidence_is_fraud
0,SYN-01025,1,0,0.60,0,0,0.98
1,SYN-01026,0,0,0.61,1,0,0.87
2,SYN-01027,0,1,0.56,0,0,0.98
3,SYN-01028,0,0,0.55,0,0,0.98
4,SYN-01029,1,0,0.73,0,0,1.00
5,SYN-01030,0,0,0.77,0,0,0.98
6,SYN-01031,0,0,0.61,0,0,0.98
7,SYN-01032,0,0,0.81,0,0,0.98


## Next step

Open `compare_saved_results.ipynb`. It uses the direct BTP result when available, and otherwise labels the supplied Playground export as the comparison source.